In [2]:
import numpy as np
import pandas as pd
import os
import random
import math
from decimal import Decimal as dec
import datetime
import time
import gc
import lightgbm as lgb
import pickle
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [3]:
## Dataframe memory reduction strategy

def reduce_mem_usage(df, verbose=True):

    # Numerical data types to consider
    numerics = ['int16', 'int32', 'int64',
                'float16', 'float32', 'float64']

    # Calculate initial memory consumption in MB
    start_mem = df.memory_usage(deep=True).sum() / 1024**2

    # Examine each column individually
    for col in df.columns:

        col_type = df[col].dtype

        # Process only numerical columns
        if col_type in numerics:

            # Find minimum and maximum values
            c_min = df[col].min()
            c_max = df[col].max()

            # Integer columns
            if str(col_type)[:3] == 'int':

                if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)

                elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)

                elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)

            # Floating-point columns
            else:

                if c_min >= np.finfo(np.float32).min and c_max <= np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)

    # Calculate final memory consumption
    end_mem = df.memory_usage(deep=True).sum() / 1024**2

    # Display memory reduction
    if verbose:
        print(
            'Mem. usage decreased to {:5.2f} Mb ({:.1f}% reduction)'
            .format(
                end_mem,
                100 * (start_mem - end_mem) / start_mem
            )
        )

    return df


In [6]:
##Load data

def load_data():
    sales_df = reduce_mem_usage(pd.read_csv("Phase 0 - Sales.csv"))
    prices_df = reduce_mem_usage(pd.read_csv("Phase 0 - Price.csv"))
    return sales_df, prices_df

In [8]:

def prepare_data(train_df, end_train_day_x, predict_horizon):

    # 1. Identify each individual time series
    index_columns = ['Client', 'Warehouse', 'Product']

    # Convert the training cutoff to datetime
    end_train_day_x = pd.Timestamp(end_train_day_x)

    # 2. Convert sales data from wide to long format
    train_df0 = pd.melt(
        train_df,
        id_vars=index_columns,
        var_name='Date',
        value_name='Sales'
    )

    # Convert observation dates to datetime
    train_df0['Date'] = pd.to_datetime(train_df0['Date'])

    # Construct a unique identifier for each time series
    train_df0['id'] = (
        train_df0['Client'].astype(str) + '-' +
        train_df0['Warehouse'].astype(str) + '-' +
        train_df0['Product'].astype(str)
    )

    # Reduce memory consumption
    train_df0 = reduce_mem_usage(train_df0, verbose=False)

    # 3. Identify the next predict_horizon weekly observation dates. This uses actual dates in the dataset rather than assuming
    # that every observation is exactly seven days apart.

    future_dates = (
        train_df0.loc[
            train_df0['Date'] > end_train_day_x, 'Date'
        ]
        .drop_duplicates()
        .sort_values()
        .head(predict_horizon)
    )
    
    if len(future_dates) < predict_horizon:
        raise ValueError("Insufficient future dates for the requested horizon.")

    # 4. Extract actual holdout observations
    time_mask = train_df0['Date'].isin(future_dates)

    val_df = (
        train_df0.loc[time_mask, ['id', 'Date', 'Sales']]
        .reset_index(drop=True)
    )

    val_path = (
        f"val_df_{end_train_day_x}_to_{future_dates.iloc[-1]}.feather"
    )

    val_df.to_feather(val_path)

    # Release holdout memory
    del val_df
    gc.collect()

    # 5. Retain historical observations only
    train_df1 = (
        train_df0.loc[
            train_df0['Date'] <= end_train_day_x
        ].copy()
    )

    # 6. Generate future prediction rows
    # Obtain unique time series combinations ONCE
    series_df = (
        train_df[index_columns]
        .drop_duplicates()
        .reset_index(drop=True)
        .copy()
    )

    # Construct the same unique series identifiers
    series_df['id'] = (
        series_df['Client'].astype(str) + '-' +
        series_df['Warehouse'].astype(str) + '-' +
        series_df['Product'].astype(str)
    )

    # Repeat each time series for every future observation date
    add_grid = (
        series_df.loc[
            series_df.index.repeat(predict_horizon)
        ].reset_index(drop=True)
    )

    # Assign future dates to the repeated series
    add_grid['Date'] = np.tile(
        future_dates.to_numpy(),
        len(series_df)
    )

    # Future sales are unknown
    add_grid['Sales'] = np.nan

    # 7. Combine historical and future observations
    grid_df = pd.concat(
        [train_df1, add_grid],
        ignore_index=True
    )

    # 8. Convert identifiers into categorical variables
    for col in index_columns + ['id']:
        grid_df[col] = grid_df[col].astype('category')

    # Reduce memory consumption
    grid_df = reduce_mem_usage(grid_df, verbose=False)

    # 9. Save the prepared grid
    grid_path = (
        f"grid_df1_{end_train_day_x}_to_{future_dates.iloc[-1]}.feather"
    )

    grid_df.to_feather(grid_path)

    # Release memory
    del grid_df, train_df0, train_df1, series_df, add_grid
    gc.collect()

    return grid_path, val_path


In [9]:
#Helper concat function to minimize memory use

def merge_by_concat(df1, df2, merge_on):

    merged_gf = df1[merge_on]

    merged_gf = merged_gf.merge(
        df2, on=merge_on, how='left'
    )

    new_columns = [
        col for col in merged_gf
        if col not in merge_on
    ]

    df1[new_columns] = merged_gf[new_columns]

    return df1

In [10]:
# Generate price features
def merge_price_features(prices_df, end_train_day_x, predict_horizon):

    end_train_day_x = pd.Timestamp(end_train_day_x)

    grid_df = pd.read_feather(
        f"grid_df1_{end_train_day_x}_to_{end_train_day_x + predict_horizon}.feather"
    )

    # Identifier columns
    index_columns = ['Client', 'Warehouse', 'Product']

    # Extract date columns
    price_columns = [
        col for col in prices_df.columns
        if col not in index_columns
    ]

    # Create feature template
    price_feature = prices_df[index_columns].copy()

    # Calculate price statistics across dates
    price_feature['price_max'] = prices_df[price_columns].max(axis=1)
    price_feature['price_min'] = prices_df[price_columns].min(axis=1)
    price_feature['price_mean'] = prices_df[price_columns].mean(axis=1)
    price_feature['price_median'] = prices_df[price_columns].median(axis=1)
    price_feature['price_std'] = prices_df[price_columns].std(axis=1)
    price_feature['max_price_diff'] = (
        price_feature['price_max'] - price_feature['price_min']
    )
    price_feature['price_nunique'] = prices_df[price_columns].nunique(axis=1)

    # Merge price features into the grid
    grid_df = merge_by_concat(
        grid_df,
        price_feature,
        index_columns
    )

    grid_df = reduce_mem_usage(grid_df)
    grid_df.to_feather(f"grid_df2_{end_train_day_x}_to_{end_train_day_x + predict_horizon}.feather")
    
    del(grid_df)
    gc.collect()


In [11]:
# Generate time features
def merge_time_features(end_train_day_x, predict_horizon):

    end_train_day_x = pd.Timestamp(end_train_day_x)

    grid_df = pd.read_feather(
        f"grid_df2_{end_train_day_x}_to_{end_train_day_x + predict_horizon}.feather"
    )

    # Identifier columns
    index_columns = ['id', 'Date']

    # Create feature template
    time_feature = grid_df[index_columns].copy()

    # Week of year
    time_feature['week_num'] = (
        grid_df['Date'].dt.isocalendar().week.astype(np.int8)
    )

    # Month of year
    time_feature['month_num'] = (
        grid_df['Date'].dt.month.astype(np.int8)
    )

    # Calendar year
    time_feature['year_num'] = grid_df['Date'].dt.year

    # Relative year index
    time_feature['diff_year'] = (
        time_feature['year_num'] - time_feature['year_num'].min()
    ).astype(np.int8)

    # Merge time features into the grid
    grid_df = merge_by_concat(
        grid_df,
        time_feature,
        index_columns
    )

    grid_df = reduce_mem_usage(grid_df)

    grid_df.to_feather(
        f"grid_df3_{end_train_day_x}_to_{end_train_day_x + predict_horizon}.feather"
    )

    del grid_df
    gc.collect()

In [ ]:

# Generate sales lag and rolling features

def merge_lag_features(end_train_day_x, predict_horizon):

    end_train_day_x = pd.Timestamp(end_train_day_x)

    forecast_end = (
        end_train_day_x + pd.Timedelta(weeks=predict_horizon)
    )

    grid_df = pd.read_feather(
        f"grid_df1_{end_train_day_x}_to_{forecast_end}.feather"
    )

    # Ensure chronological ordering within each series
    grid_df = grid_df.sort_values(
        ['id', 'Date']
    ).reset_index(drop=True)

    grid_df = grid_df[['id','Date']]
    # Individual lag features
    num_lag_day = 15

    num_lag_day_list = (
        list(range(
            predict_horizon,
            predict_horizon + num_lag_day
        ))
        + [52]
    )

    for lag in num_lag_day_list:

        grid_df[f'lag_{lag}'] = (
            grid_df.groupby('id')['Sales'].shift(lag)
        )

    # Rolling features on Sales
    num_rolling_day_list = [4, 8, 13, 26, 52]

    for window in num_rolling_day_list:

        grid_df[f'rolling_mean_{window}'] = (
            grid_df.groupby('id')['Sales'].transform(
                lambda x: x.shift(predict_horizon)
                           .rolling(window).mean()
            )
        )

        grid_df[f'rolling_std_{window}'] = (
            grid_df.groupby('id')['Sales'].transform(
                lambda x: x.shift(predict_horizon)
                           .rolling(window).std()
            )
        )

    # Additional rolling features on lag_52
    num_lag52_rolling_list = [4, 8, 13]

    for window in num_lag52_rolling_list:

        grid_df[f'lag_52_rolling_mean_{window}'] = (
            grid_df.groupby('id')['lag_52'].transform(
                lambda x: x.rolling(window).mean()
            )
        )

        grid_df[f'lag_52_rolling_median_{window}'] = (
            grid_df.groupby('id')['lag_52'].transform(
                lambda x: x.rolling(window).median()
            )
        )

        grid_df[f'lag_52_rolling_std_{window}'] = (
            grid_df.groupby('id')['lag_52'].transform(
                lambda x: x.rolling(window).std()
            )
        )

    # Reduce feature memory usage
    feature_cols = [
        col for col in grid_df.columns
        if col.startswith('lag_') or col.startswith('rolling_')
    ]

    grid_df[feature_cols] = grid_df[feature_cols].astype(
        np.float16
    )

    grid_df = reduce_mem_usage(grid_df, verbose=False)

    grid_df.to_feather(
        f"lag_features_{end_train_day_x}_to_{forecast_end}.feather"
    )

    del grid_df
    gc.collect()


In [ ]:

# Target encoding

def merge_target_encoding_feature(end_train_day_x, predict_horizon):

    end_train_day_x = pd.Timestamp(end_train_day_x)

    forecast_end = (
        end_train_day_x + pd.Timedelta(weeks=predict_horizon)
    )

    grid_df = pd.read_feather(
        f"grid_df1_{end_train_day_x}_to_{forecast_end}.feather"
    )

    # Mask the most recent predict_horizon weeks of sales
    cutoff_date = (
        end_train_day_x - pd.Timedelta(weeks=predict_horizon)
    )

    grid_df.loc[grid_df['Date'] > cutoff_date, 'Sales'] = np.nan

    base_cols = list(grid_df)

    icols = [
        ['Client'],
        ['Warehouse'],
        ['Product'],
        ['Client', 'Warehouse'],
        ['Client', 'Product'],
        ['Warehouse', 'Product']
    ]

    for col in icols:

        col_name = '_' + '_'.join(col) + '_'

        grid_df['enc' + col_name + 'mean'] = (
            grid_df.groupby(col)['Sales']
            .transform('mean')
            .astype(np.float16)
        )

        grid_df['enc' + col_name + 'std'] = (
            grid_df.groupby(col)['Sales']
            .transform('std')
            .astype(np.float16)
        )

    keep_cols = [
        col for col in grid_df.columns
        if col not in base_cols
    ]

    grid_df = grid_df[['id', 'Date'] + keep_cols]

    grid_df = reduce_mem_usage(grid_df, verbose=False)

    grid_df.to_feather(
        f"target_encoding_{end_train_day_x}_to_{forecast_end}.feather"
    )

    del grid_df
    gc.collect()


In [ ]:

def assemble_grid(end_train_day_x, predict_horizon):

    end_train_day_x = pd.Timestamp(end_train_day_x)

    forecast_end = (
        end_train_day_x + pd.Timedelta(weeks=predict_horizon)
    )

    # Load base grid with price and time features
    grid_df = pd.read_feather(
        f"grid_df3_{end_train_day_x}_to_{forecast_end}.feather"
    )

    # Add target-encoding features
    target_df = pd.read_feather(
        f"target_encoding_{end_train_day_x}_to_{forecast_end}.feather"
    )

    assert grid_df[['id', 'Date']].equals(
        target_df[['id', 'Date']]
    )

    grid_df = pd.concat(
        [grid_df, target_df.iloc[:, 2:]],
        axis=1
    )

    del target_df

    # Add lag features
    lag_df = pd.read_feather(
        f"lag_feature_{end_train_day_x}_to_{forecast_end}.feather"
    )

    assert grid_df[['id', 'Date']].equals(
        lag_df[['id', 'Date']]
    )

    grid_df = pd.concat(
        [grid_df, lag_df.iloc[:, 2:]],
        axis=1
    )

    del lag_df

    # Reduce memory usage
    grid_df = reduce_mem_usage(grid_df, verbose=False)

    # Save assembled grid
    grid_df.to_feather(
        f"grid_full_{end_train_day_x}_to_{forecast_end}.feather"
    )

    gc.collect()

    return grid_df